# 하천 쓰레기 분류 AI 프로그램 구현 · 4차시

**사용법**
1. 메뉴 **파일 → Drive에 사본 저장**을 눌러 내 노트북으로 만듭니다.
2. 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU**를 고릅니다. (없으면 CPU로도 됩니다)
3. **⓪ 실험 설정** 칸만 바꾸고, 위에서부터 셀을 차례로 실행(▶)합니다.
4. 처음에는 아무것도 바꾸지 않고 실행해 **기준 모델**의 테스트 점수(⑥, 8장 중 맞힌 수)를 확인합니다. 3차시 티처블 머신 점수와 비교해 보세요.
5. 개선 전략 하나를 `True`로 바꾸고 **⓪부터 다시 실행**해 ⑦ 기록표에서 점수를 비교합니다.

## ⓪ 실험 설정 — 이 칸만 바꿉니다!

In [ ]:
MY_NAME = "10301 김공주"      # 내 학번 이름

# 개선 전략: 한 번에 하나만 True 로 바꿉니다
ADD_DIRTY_PHOTOS = False   # 데이터 보강: 흙 묻은 사진 80장을 훈련 데이터에 추가
USE_AUGMENTATION = False   # 데이터 증강: 사진을 돌리고·확대하고·뒤집어서 여러 모습으로 보여 주기
USE_TRANSFER     = False   # 전이 학습: 사진을 잘 보는 모델(MobileNetV2)을 빌려 쓰기

NUM_EPOCHS    = 50       # 최대 학습 횟수
LEARNING_RATE = 0.0002   # AI가 학습할 때 나아가는 '보폭'

strategy = [n for n, v in [("데이터 보강", ADD_DIRTY_PHOTOS), ("데이터 증강", USE_AUGMENTATION), ("전이 학습", USE_TRANSFER)] if v]
if len(strategy) > 1:
    print("⚠️ 개선 전략은 한 번에 하나만 켜야 무엇 때문에 결과가 달라졌는지 알 수 있어요!")
print("이번 실험:", strategy[0] if strategy else "기준 모델 (아무것도 바꾸지 않음)")

## ① 필요한 라이브러리 불러오기

In [ ]:
import os, urllib.request, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow Version:", tf.__version__)

## ② 데이터 준비하기
쓰레기 사진을 받아 와서 **훈련 데이터 80%**와 **검증 데이터 20%**로 나눕니다.
(교재처럼 구글 드라이브를 쓰지 않고, 선생님이 올려 둔 사진을 자동으로 받습니다)

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/moonyoungwoong/2026divdetreshfromriver/main/colab/river_trash_data.zip"
if not os.path.exists("river_trash"):
    urllib.request.urlretrieve(DATA_URL, "river_trash.zip")
    zipfile.ZipFile("river_trash.zip").extractall(".")

DATA_DIR  = "river_trash/trash_data"     # 클래스당 50장 (깨끗한 사진)
DIRTY_DIR = "river_trash/dirty_extra"    # 데이터 보강용 흙 묻은 사진
TEST_DIR  = "river_trash/test"           # 3차시와 같은 공통 테스트 사진 8장

IMAGE_SIZE = (128, 128)   # 모든 사진을 가로 128, 세로 128 픽셀로 맞춤
BATCH_SIZE = 16           # 한 번에 AI에 넣는 사진 수
SEED = 123                # 무작위로 나눌 때 모두 같은 결과가 나오게 하는 값
tf.keras.utils.set_random_seed(SEED)

train_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="training", seed=SEED,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE)
validation_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="validation", seed=SEED,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE)
CLASS_NAMES = train_dataset.class_names
print("분류 클래스:", CLASS_NAMES)

if ADD_DIRTY_PHOTOS:   # 흙 묻은 사진은 훈련 데이터에만 더한다 (검증 데이터는 그대로 두어 공정하게 비교)
    dirty_dataset = tf.keras.utils.image_dataset_from_directory(
        DIRTY_DIR, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, class_names=CLASS_NAMES, seed=SEED)
    train_dataset = train_dataset.concatenate(dirty_dataset).shuffle(64, seed=SEED)
    print("➕ 흙 묻은 사진을 훈련 데이터에 추가했습니다.")

AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.cache().prefetch(buffer_size=AUTOTUNE)

## ③ 모델 만들기
- 기본: 교재의 **CNN** (합성곱층으로 사진의 특징을 찾고, 완전 연결층으로 최종 분류)
- 전이 학습: 100만 장 넘는 사진으로 미리 학습된 **MobileNetV2**가 특징을 찾고, 마지막 판단만 새로 학습

In [ ]:
tf.keras.utils.set_random_seed(SEED)
input_shape = IMAGE_SIZE + (3,)
num_classes = len(CLASS_NAMES)

augment = [
    layers.RandomRotation(0.2, seed=SEED),
    layers.RandomZoom(0.2, seed=SEED),
    layers.RandomFlip("horizontal", seed=SEED),
] if USE_AUGMENTATION else []

if USE_TRANSFER:
    base = tf.keras.applications.MobileNetV2(input_shape=input_shape, include_top=False, weights="imagenet")
    base.trainable = False   # 빌려 온 '눈'은 그대로 두고
    model = models.Sequential([
        layers.Input(shape=input_shape),
        *augment,
        layers.Rescaling(1./127.5, offset=-1),   # MobileNetV2가 원하는 -1 ~ 1 범위로 맞춤
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation="softmax"),   # 마지막 판단만 새로 학습
    ])
else:
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Rescaling(1./255),
        *augment,
        # 합성곱층: 이미지에서 특징 추출
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),          # 2D 데이터를 1D로 펼치기
        layers.Dropout(0.6),
        # 완전 연결층: 추출된 특징을 기반으로 최종 분류
        layers.Dense(128, activation="relu"),
        layers.Dense(num_classes, activation="softmax"),
    ])
model.summary()

## ④ 학습 방법 정하기 (컴파일)

In [ ]:
model.compile(optimizer=Adam(learning_rate=LEARNING_RATE),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=["accuracy"])

# 검증 손실이 10번 연속 나아지지 않으면 멈추고, 가장 좋았던 때로 되돌림 (과적합 방지)
early_stop = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, verbose=1)

## ⑤ 학습시키고 그래프 보기

In [ ]:
print(f"--- AI 학습 시작 (Learning Rate: {LEARNING_RATE}, Max Epochs: {NUM_EPOCHS}) ---")
history = model.fit(train_dataset, validation_data=validation_dataset,
                    epochs=NUM_EPOCHS, callbacks=[early_stop], verbose=2)
print("--- AI 학습 완료 ---")

h = history.history
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(h["accuracy"], label="train acc", color="#5170FF")
ax[0].plot(h["val_accuracy"], label="val acc", color="#FF3131")
ax[0].set_title("Accuracy"); ax[0].set_xlabel("Epochs"); ax[0].set_ylim(0, 1.05); ax[0].legend()
ax[1].plot(h["loss"], label="train loss", color="#5170FF")
ax[1].plot(h["val_loss"], label="val loss", color="#FF3131")
ax[1].set_title("Loss"); ax[1].set_xlabel("Epochs"); ax[1].legend()
plt.show()

best = int(np.argmin(h["val_loss"]))
val_acc = h["val_accuracy"][best]
print(f"가장 좋았던 에포크: {best + 1}번째 · 그때 훈련 정확도 {h['accuracy'][best]:.0%}, 검증 정확도 {val_acc:.0%}")

## ⑥ 공통 테스트 사진 8장으로 채점하기
3차시 티처블 머신 때와 **같은 8장**입니다. 짝수 번호는 흙 묻은 사진입니다.

In [ ]:
TEST_ANSWERS = {1: "plastic", 2: "plastic", 3: "can", 4: "can", 5: "glass", 6: "glass", 7: "other", 8: "other"}
rows, fig = [], plt.figure(figsize=(16, 4.5))
for no, answer in TEST_ANSWERS.items():
    img = tf.keras.utils.load_img(f"{TEST_DIR}/{no}.jpg", target_size=IMAGE_SIZE)
    prob = model.predict(np.expand_dims(tf.keras.utils.img_to_array(img), 0), verbose=0)[0]
    pred = CLASS_NAMES[int(np.argmax(prob))]
    ok = pred == answer
    rows.append({"사진": f"{no}" + (" (흙)" if no % 2 == 0 else ""), "실제값": answer, "AI 예측값": pred,
                 "확률": f"{prob.max():.0%}", "결과": "⭕" if ok else "❌"})
    a = fig.add_subplot(1, 8, no); a.imshow(img); a.axis("off")
    a.set_title(f"#{no} answer: {answer}\nAI: {pred} {prob.max():.0%}", color="#1D1D1F" if ok else "#FF3131", fontsize=9)
plt.show()

test_df = pd.DataFrame(rows)
score = int((test_df["결과"] == "⭕").sum())
clean = int(sum(r["결과"] == "⭕" for i, r in enumerate(rows) if i % 2 == 0))
dirty = score - clean
display(test_df)
print(f"테스트 점수: {score} / 8  (깨끗한 사진 {clean}/4, 흙 묻은 사진 {dirty}/4)")

## ⑦ 내 실험 기록표
실행할 때마다 한 줄씩 쌓입니다. **기준 모델 → 개선 후**를 비교해 개선 계획서와 발표에 씁니다.

In [ ]:
try:
    RESULTS
except NameError:
    RESULTS = []
RESULTS.append({
    "이름": MY_NAME,
    "실험": strategy[0] if strategy else "기준 모델",
    "에포크(최대/실제)": f"{NUM_EPOCHS}/{len(h['loss'])}",
    "학습률": LEARNING_RATE,
    "검증 정확도": f"{val_acc:.0%}",
    "테스트 점수(/8)": score,
    "흙 묻은 사진(/4)": dirty,
})
display(pd.DataFrame(RESULTS))